# FallVision 05: final evaluation of the submitted system

Pipeline step 3 of 3. Runs exactly the `infer.py` engine (`src/fv_engine.py`: frame by frame, current
and past frames only) on every validation and test video with `src/eval_causal.py`, through
`run_pipeline.py`:

- decision lags 0 / 0.5 / 1.0 / 1.5 s scored; the lag (best average precision) and the alarm threshold
  (best F1, ties -> fewer false alarms) are chosen on **validation only**, then every test set is
  reported once -> `results/causal_eval.md`;
- the chosen lag and threshold are written into `config.yaml`;
- finally `infer.py` itself (the evaluation command) is run on two test videos as a check.

**Before running (Kaggle)**
- Inputs: the outputs of `01_download_omnifall_videos`, `02_download_gmdcsa24` and `04_train_models`;
  optional: the private dataset with our own videos (`fallvision-own-videos`).
- *Settings*: Accelerator **GPU T4 x1**, Internet **on**. *Add-ons > Secrets*: `GITHUB_TOKEN` on.
- Time: about 2 hours. **Save Version > Save & Run All (Commit)**.

In [ ]:
# ---- settings: the only cell you may need to change ----
REPO_URL = "https://github.com/Prish1509/project_fallvision.git"   # your repository (public)
BRANCH = "main"
PUSH_RESULTS = True             # push the result files back to the repository (needs the GITHUB_TOKEN secret)
REPO = "/kaggle/working/project_fallvision"   # where the code is cloned in this notebook
WORK = "/kaggle/working/work"         # pose caches, features, windows, probabilities (kept in the Output)

## 1. Code and packages

In [ ]:
import os, shutil, subprocess, sys
os.makedirs("/kaggle/working", exist_ok=True)
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
import kaggle_helpers as kh

# the exact versions from requirements.txt of the packages Kaggle does not have (torch is Kaggle's own)
pins = [l.split("#")[0].strip() for l in open("requirements.txt")
        if l.split("==")[0].strip() in ("ultralytics", "ultralytics-thop", "lap")]
kh.run([sys.executable, "-m", "pip", "install", "-q", *pins])

import torch
print("code version:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - turn it on: Settings > Accelerator > GPU T4 x1")

## 2. Inputs: videos, the model trained by notebook 04, the pretrained pose model

In [ ]:
OMNI = kh.find_omnifall()
GMD = kh.find_gmd()
assert OMNI and GMD, "add the outputs of the two data notebooks as inputs (see the top of this notebook)"
OWN = kh.find_own(skip=(OMNI, GMD))
os.makedirs("models", exist_ok=True)            # models/ is not in git: weights are never committed
shutil.copyfile(kh.find_trained_model(), "models/gru_hard.pt")
kh.get_yolo("models/yolo11m-pose.pt")
kh.env_report("results/env_kaggle.txt")

## 3. Evaluation (validation and every test set)

In [ ]:
kh.run([sys.executable, "run_pipeline.py", "--omnifall-root", OMNI, "--gmd-root", GMD,
        *(["--own-root", OWN] if OWN else []), "--work", WORK, "--from", "evaluate", "--to", "evaluate"])
shutil.copyfile(f"{WORK}/causal_scores.csv", "results/causal_scores.csv")

In [ ]:
print("Chosen setting per split, committed version -> this run:\n")
kh.compare_eval(kh.git_text(REPO, "results/causal_eval.md"), open("results/causal_eval.md").read())
print()
kh.show_change(REPO, "config.yaml")

## 4. Check: the evaluation command (`infer.py`) on two test videos

In [ ]:
import csv, json
from fv_engine import load_config
from cache_poses import video_file
thr = load_config("config.yaml")["threshold"]
rows = [r for r in csv.DictReader(open(f"{WORK}/causal_scores.csv")) if r["split"] == "test_gmdcsa24"]
fall = max((r for r in rows if r["has_fall"] == "1"), key=lambda r: float(r["max_score"]))     # a fall it catches
nofall = min((r for r in rows if r["has_fall"] == "0"), key=lambda r: float(r["max_score"]))   # a clear non-fall
CHECK = "/kaggle/working/infer_check"
shutil.rmtree(CHECK, ignore_errors=True)
os.makedirs(f"{CHECK}/videos")
expected = {}
for kind, r in (("fall", fall), ("nofall", nofall)):
    name = f"{kind}_" + r["path"].replace("/", "_")
    shutil.copyfile(video_file(r["dataset"], r["path"], OMNI, GMD), f"{CHECK}/videos/{name}.mp4")
    expected[name] = int(float(r["max_score"]) >= thr)
kh.run([sys.executable, "infer.py", "--source", f"{CHECK}/videos", "--config", "config.yaml",
        "--output-dir", f"{CHECK}/results", "--device", "cuda:0", "--no-display"])
summary = json.load(open(f"{CHECK}/results/run_summary.json"))
alarms = [json.loads(l) for l in open(f"{CHECK}/results/alarms.jsonl") if l.strip()]
print("status:", summary["status"], "| frames processed:", summary["frames_processed"],
      "| frames/s (including model loading):", round(summary["frames_processed"] / max(summary["total_runtime_s"], 1e-9), 1))
for name, exp in expected.items():
    got = sum(a["source_id"] == name for a in alarms)
    print(f"{name}: alarms expected from the evaluation {exp}, infer.py gave {got}", "OK" if min(got, 1) == exp else "DIFFERENT")
for a in alarms:
    print(json.dumps(a))

## 5. Save the results to GitHub

In [ ]:
if PUSH_RESULTS:
    kh.push(REPO, ["results/causal_eval.md", "results/causal_scores.csv", "results/env_kaggle.txt", "config.yaml"],
            "Pipeline 05: final evaluation; decision lag and threshold chosen on validation", REPO_URL, BRANCH)